# OmniHC Colab setup

Run the environment cell once, then run the data cell whenever you want to switch the benchmark dataset copied from Drive.

In [ ]:
# Environment setup
import os
import sys
from pathlib import Path

try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook setup cell is intended for Google Colab.") from exc

drive.mount("/content/drive")

REPO_URL = "https://github.com/duasob/omni_hc.git"
PROJECT_ROOT = Path("/content/omni_hc")
GH_TOKEN = userdata.get("GH_TOKEN")

clone_url = REPO_URL
if GH_TOKEN:
    clone_url = REPO_URL.replace("https://", f"https://oauth2:{GH_TOKEN}@")

if PROJECT_ROOT.exists():
    %cd /content/omni_hc
    !git pull --ff-only
else:
    !git clone --recursive {clone_url} {PROJECT_ROOT}
    %cd /content/omni_hc

!pip install -q -e ".[experiments]"

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

os.environ.setdefault("OMNI_HC_PROJECT_ROOT", str(PROJECT_ROOT))

# Symlink outputs → Drive so checkpoints persist across sessions
drive_outputs = Path("/content/drive/MyDrive/Y4_bulk/omni_hc/outputs")
drive_outputs.mkdir(parents=True, exist_ok=True)
outputs_link = PROJECT_ROOT / "outputs"
if outputs_link.is_symlink():
    outputs_link.unlink()
if not outputs_link.exists():
    os.symlink(drive_outputs, outputs_link)
print(f"outputs → {drive_outputs}")

print(f"Project ready at {PROJECT_ROOT}")

In [ ]:
# Copy one or all benchmark datasets from Google Drive into the repo data directory.
import shutil
import zipfile
from pathlib import Path

# Options: "darcy", "navier_stokes", "pipe", "elasticity", "plasticity", "all"
BENCHMARK = "all"

# Override in Colab secrets if your Drive folder differs.
# Expected layout: <DRIVE_DATA_ROOT>/<dataset>.zip or <DRIVE_DATA_ROOT>/<dataset>/
drive_data_root_secret = userdata.get("DRIVE_DATA_ROOT")
if drive_data_root_secret:
    DRIVE_DATA_ROOT = Path(drive_data_root_secret)
else:
    drive_roots = [
        Path("/content/drive/MyDrive/omni_hc/data"),
        Path("/content/drive/MyDrive/hc_fluid/data"),
    ]
    DRIVE_DATA_ROOT = next((root for root in drive_roots if root.exists()), drive_roots[0])

DATASETS = {
    "darcy": {
        "drive_name": "Darcy_421",
        "dest": Path("data/Darcy_421"),
        "required": ["piececonst_r421_N1024_smooth1.mat", "piececonst_r421_N1024_smooth2.mat"],
    },
    "navier_stokes": {
        "drive_name": "NavierStokes_V1e-5_N1200_T20",
        "dest": Path("data/NavierStokes_V1e-5_N1200_T20"),
        "required": ["NavierStokes_V1e-5_N1200_T20.mat"],
    },
    "pipe": {
        "drive_name": "pipe",
        "dest": Path("data/pipe"),
        "required": ["Pipe_X.npy", "Pipe_Y.npy", "Pipe_Q.npy"],
    },
    "elasticity": {
        "drive_name": "elasticity",
        "dest": Path("data/elasticity"),
        "required": ["Random_UnitCell_sigma_10.npy", "Random_UnitCell_XY_10.npy"],
    },
    "plasticity": {
        "drive_name": "plasticity",
        "dest": Path("data/plasticity"),
        "required": ["plas_N987_T20.mat"],
    },
}

def contains_required(root: Path, required: list[str]) -> bool:
    direct = all((root / name).exists() for name in required)
    meshes = all((root / "Meshes" / name).exists() for name in required)
    return direct or meshes

def copy_payload(src: Path, dest: Path, required: list[str]) -> None:
    candidates = [src]
    if src.is_dir():
        candidates.extend(path for path in src.iterdir() if path.is_dir())
    payload = next((path for path in candidates if contains_required(path, required)), src)
    dest.mkdir(parents=True, exist_ok=True)
    if payload.is_file():
        shutil.copy2(payload, dest / payload.name)
        return
    shutil.copytree(payload, dest, dirs_exist_ok=True)

def zip_directory(source_dir: Path, zip_path: Path) -> None:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_zip_path = zip_path.with_suffix(zip_path.suffix + ".tmp")
    if tmp_zip_path.exists():
        tmp_zip_path.unlink()
    with zipfile.ZipFile(tmp_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in source_dir.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(source_dir.parent))
    tmp_zip_path.replace(zip_path)


def load_dataset(name: str) -> None:
    spec = DATASETS[name]
    dest = PROJECT_ROOT / spec["dest"]
    required = spec["required"]

    if contains_required(dest, required):
        print(f"[{name}] already present at {dest}")
        return

    drive_name = spec["drive_name"]
    source_dir = DRIVE_DATA_ROOT / drive_name
    source_zip = DRIVE_DATA_ROOT / f"{drive_name}.zip"
    tmp_extract = Path("/content") / f"{drive_name}_extract"

    if not source_zip.exists():
        if source_dir.exists():
            print(f"[{name}] creating Drive cache zip at {source_zip}")
            zip_directory(source_dir, source_zip)
        else:
            raise FileNotFoundError(
                f"[{name}] could not find {source_zip} or {source_dir}. "
                "Set DRIVE_DATA_ROOT in Colab secrets if your Drive layout is different."
            )

    shutil.rmtree(tmp_extract, ignore_errors=True)
    tmp_extract.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(source_zip) as archive:
            archive.extractall(tmp_extract)
    except zipfile.BadZipFile as exc:
        raise FileNotFoundError(
            f"[{name}] found {source_zip}, but it is not a valid zip. Delete it and rerun this cell."
        ) from exc
    copy_payload(tmp_extract, dest, required)
    shutil.rmtree(tmp_extract, ignore_errors=True)

    if not contains_required(dest, required):
        raise FileNotFoundError(
            f"[{name}] copied data to {dest}, but required files were not found: {required}"
        )
    print(f"[{name}] loaded at {dest}")


targets = list(DATASETS.keys()) if BENCHMARK == "all" else [BENCHMARK]
if BENCHMARK != "all" and BENCHMARK not in DATASETS:
    raise KeyError(
        f"Unknown BENCHMARK={BENCHMARK!r}. Options: {sorted(DATASETS) + ['all']}"
    )

for name in targets:
    load_dataset(name)

print(f"\nDone: {', '.join(targets)}")

In [14]:
!git pull

Already up to date.


In [13]:
folder_path = "/content/drive/MyDrive/Y4_bulk/omni_hc/outputs/darcy/none/transolver/final/seed_42/"
!python scripts/test.py \
  --config {folder_path}resolved_config.yaml \
  --checkpoint {folder_path}latest.pt

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
test_mse=0.000000 test_rel_l2=0.009170
test_media:
  coeff: /content/drive/MyDrive/Y4_bulk/omni_hc/outputs/darcy/none/transolver/final/seed_42/test_media/test_coeff.png
  pred: /content/drive/MyDrive/Y4_bulk/omni_hc/outputs/darcy/none/transolver/final/seed_42/test_media/test_pred.png
  target: /content/drive/MyDrive/Y4_bulk/omni_hc/outputs/darcy/none/transolver/final/seed_42/test_media/test_target.png
  error_signed: /content/drive/MyDrive/Y4_bulk/omni_hc/outputs/darcy/none/transolver/final/seed_42/test_media/test_error_signed.png
  error_abs: /content/drive/MyDrive/Y4_bulk/omni_hc/outputs/darcy/none/transolver/final/seed_42/test_media/test_error_abs.png
  fields: /content/drive/MyDrive/Y4_bulk/omni_hc/outputs/darcy